# MLIP Training Data Preparation Toolkit

A comprehensive Python toolkit for preparing machine learning interatomic potential (MLIP) training data from VASP calculations.

## Features

- **OUTCAR to CFG Conversion**: Extract atomic configurations, forces, energies, and stresses from VASP OUTCAR files
- **Energy Convergence Analysis**: Analyze and visualize energy convergence during VASP calculations
- **Data Filtering**: Filter configurations based on convergence criteria
- **File Management**: Merge multiple CFG files and manage large datasets
- **Grade Analysis**: Analyze MLIP sampling grades and visualize distributions

## Quick Start

1. **Update file paths** in the example cells to match your data location
2. **Run the import cell** first to load all functions
3. **Choose the workflow** you need from the sections below:
   - Convert VASP OUTCAR → MLIP CFG format
   - Analyze energy convergence
   - Filter and merge datasets
   - Analyze MLIP grades

## CFG Format Details

The MLIP CFG format contains:
- **Atomic positions** and **forces** (in Cartesian coordinates)
- **Lattice vectors** (supercell definition)
- **Total energy** (in eV)
- **Stress tensor** (converted from kB to eV using cell volume)
- **Atom types** (0 for B, 1 for Ti)

---

In [ ]:
# =============================================================================
# IMPORTS AND CORE FUNCTIONS
# =============================================================================
import numpy as np  
import re  
import matplotlib.pyplot as plt
import os
import glob

print("✓ All libraries imported successfully!")
print("Ready to process VASP data for MLIP training")

---

# 1. VASP OUTCAR to MLIP CFG Conversion

## 1.1 Core Conversion Functions

Convert VASP OUTCAR files to MLIP-compatible CFG format with proper stress tensor handling and atom type mapping.

### Key Features:
- **Automatic atom detection** from POTCAR information
- **Stress conversion** from kB to eV (multiplied by cell volume)
- **Atom type mapping** (B → 0, Ti → 1)
- **Handles multiple ionic steps** from single OUTCAR file

In [ ]:
# =============================================================================
# OUTCAR TO CFG CONVERSION FUNCTIONS
# =============================================================================

def parse_outcar_to_mlip(outcar_file, output_file):
    """
    Main function to parse VASP OUTCAR file and convert to MLIP CFG format
    
    Parameters:
    outcar_file (str): Path to VASP OUTCAR file
    output_file (str): Path for output CFG file
    
    Returns:
    int: Number of configurations successfully processed
    """
    print(f"Starting to parse {outcar_file}")

    # Read the entire OUTCAR file
    with open(outcar_file, 'r') as f:
        lines = f.readlines()

    # Extract cell volume from OUTCAR
    cell_volume = extract_cell_volume(lines)

    # Find all ionic step boundaries
    ionic_step_indices = []
    for i, line in enumerate(lines):
        if "--------------------------------------- Ionic step" in line:
            ionic_step_number = int(line.split("Ionic step")[1].split("---")[0].strip())
            ionic_step_indices.append((i, ionic_step_number))

    print(f"Found {len(ionic_step_indices)} ionic steps")

    # Process each ionic step
    valid_configs = []

    for idx, (start_line, step_num) in enumerate(ionic_step_indices):
        # Determine the end line (next ionic step or end of file)
        end_line = len(lines)
        if idx < len(ionic_step_indices) - 1:
            end_line = ionic_step_indices[idx + 1][0]

        # Extract the lines for this ionic step
        step_lines = lines[start_line:end_line]

        # Check if this ionic step contains iterations
        has_iterations = False
        ediff_reached_idx = -1
        for i, line in enumerate(step_lines):
            if "--------------------------------------- Iteration" in line:
                has_iterations = True
            if "------------------------ aborting loop because EDIFF is reached ----------------------------------------" in line:
                ediff_reached_idx = i
                break

        # If has iterations, start from EDIFF reached line
        if has_iterations and ediff_reached_idx != -1:
            effective_start = ediff_reached_idx
        else:
            effective_start = 0

        # Extract configuration data from this step
        config = extract_configuration_from_step(step_lines[effective_start:], lines, start_line + effective_start, cell_volume)
        if config:
            # Convert atom types to integers (B/Ti → 0/1)
            config['atom_types'] = convert_atom_types_to_int(config['atom_types'])
            valid_configs.append(config)
            if len(valid_configs) % 100 == 0:
                print(f"Processed {len(valid_configs)} valid configurations")
    
    print(f"Successfully extracted {len(valid_configs)} valid configurations")

    # Write configurations to CFG format
    write_mlip_cfg_file(valid_configs, output_file)

    return len(valid_configs)

def extract_cell_volume(lines):
    """Extract cell volume from OUTCAR lines."""
    for line in lines:
        if "  volume of cell :" in line:
            try:
                return float(line.split(':')[1].strip().split()[0])
            except:
                continue
    return None

def convert_atom_types_to_int(atom_types):
    """Convert atom types to integers: B→0, Ti→1"""
    type_map = {'B': 0, 'Ti': 1}
    return [type_map.get(t, -1) for t in atom_types]

def extract_configuration_from_step(step_lines, all_lines, global_offset, cell_volume):
    """Extract atomic configuration data from a single ionic step"""
    config = {
        'positions': [],
        'forces': [],
        'lattice': None,
        'energy': None,
        'stress': None,
        'atom_types': None,
        'natoms': 0
    }

    # Find positions and forces section
    pos_idx = -1
    for i, line in enumerate(step_lines):
        if "POSITION" in line and "TOTAL-FORCE" in line:
            pos_idx = i
            break

    if pos_idx == -1:
        return None

    # Extract positions and forces
    positions = []
    forces = []
    i = pos_idx + 2  # Skip header lines
    while i < len(step_lines) and "---" not in step_lines[i]:
        parts = step_lines[i].split()
        if len(parts) >= 6:
            pos = [float(parts[0]), float(parts[1]), float(parts[2])]
            positions.append(pos)
            force = [float(parts[3]), float(parts[4]), float(parts[5])]
            forces.append(force)
        i += 1

    if not positions:
        return None

    config['positions'] = positions
    config['forces'] = forces
    config['natoms'] = len(positions)

    # Extract lattice vectors
    for i in range(pos_idx, -1, -1):
        if i < len(step_lines) and "direct lattice vectors" in step_lines[i]:
            lattice = []
            for k in range(1, 4):
                if i + k < len(step_lines):
                    vector = step_lines[i + k].split()[:3]
                    lattice.append([float(x) for x in vector])
            config['lattice'] = lattice
            try:
                cell_volume = abs(np.linalg.det(np.array(lattice)))
                config['cell_volume'] = cell_volume
            except:
                config['cell_volume'] = None
            break

    # If lattice not found, look further back
    if config['lattice'] is None:
        abs_pos = global_offset + pos_idx
        for j in range(abs_pos, max(0, abs_pos - 1000), -1):
            if j < len(all_lines) and "direct lattice vectors" in all_lines[j]:
                lattice = []
                for k in range(1, 4):
                    if j + k < len(all_lines):
                        vector = all_lines[j + k].split()[:3]
                        lattice.append([float(x) for x in vector])
                config['lattice'] = lattice
                try:
                    cell_volume = abs(np.linalg.det(np.array(lattice)))
                    config['cell_volume'] = cell_volume
                except:
                    config['cell_volume'] = None
                break

    # Extract energy
    for i, line in enumerate(step_lines):
        if "total energy   ETOTAL" in line and "eV" in line:
            try:
                parts = line.split("=")
                if len(parts) > 1:
                    energy_part = parts[1].split()[0]
                    config['energy'] = float(energy_part)
                    break
            except:
                pass

    # Extract stress tensor and convert kB → eV
    for i, line in enumerate(step_lines):
        if "in kB" in line and "external pressure" not in line:
            try:
                parts = line.split()
                if len(parts) >= 8 and parts[0] == "in" and parts[1] == "kB":
                    stress_values = [float(x) for x in parts[2:8]]
                    # VASP order: XX YY ZZ XY YZ ZX
                    # MLIP order: XX YY ZZ YZ XZ XY
                    xx, yy, zz, xy, yz, zx = stress_values
                    
                    # Convert kB to eV/Å³ then multiply by volume to get eV
                    if cell_volume:
                        xx_eV = (xx / 160.2176) * cell_volume
                        yy_eV = (yy / 160.2176) * cell_volume
                        zz_eV = (zz / 160.2176) * cell_volume
                        yz_eV = (yz / 160.2176) * cell_volume
                        zx_eV = (zx / 160.2176) * cell_volume
                        xy_eV = (xy / 160.2176) * cell_volume
                        config['stress'] = [xx_eV, yy_eV, zz_eV, yz_eV, zx_eV, xy_eV]
                    break
            except:
                pass

    # Extract atom types
    config['atom_types'] = get_atom_types(all_lines, global_offset, config['natoms'])

    return config

def get_atom_types(lines, current_idx, natoms):
    """Extract atom types from POTCAR information"""
    atom_types = []
    atom_counts = []

    for i in range(min(current_idx, len(lines))):
        if "POTCAR:" in lines[i] and "PAW" in lines[i]:
            element = lines[i].split()[2].split('_')[0]
            if element not in atom_types:
                atom_types.append(element)

        if "ions per type" in lines[i]:
            counts = [int(x) for x in lines[i].split()[4:]]
            atom_counts = counts
            break

    # Create atom type list for each atom
    atom_list = []
    for atom_type, count in zip(atom_types, atom_counts):
        atom_list.extend([atom_type] * count)

    if len(atom_list) != natoms:
        return ["X"] * natoms  # Fallback for unknown types
    return atom_list

def write_mlip_cfg_file(configurations, output_file):
    """Write configurations to MLIP CFG format"""
    with open(output_file, 'w') as f:
        for i, config in enumerate(configurations):
            f.write("BEGIN_CFG\n")
            f.write("Size\n")
            f.write(f"{config['natoms']}\n")

            # Write lattice vectors
            if config['lattice']:
                f.write("Supercell\n")
                for row in config['lattice']:
                    f.write(f"{row[0]:>12.6f} {row[1]:>12.6f} {row[2]:>12.6f}\n")

            # Write atom data
            f.write("AtomData: id type cartes_x cartes_y cartes_z fx fy fz\n")
            for j, (pos, force, atom_type) in enumerate(zip(config['positions'], config['forces'], config['atom_types'])):
                f.write(f"  {j+1:>12d} {atom_type:>12d} {pos[0]:>12.8f} {pos[1]:>12.8f} {pos[2]:>12.8f} "
                       f"{force[0]:>12.8f} {force[1]:>12.8f} {force[2]:>12.8f}\n")

            # Write energy
            if config['energy'] is not None:
                f.write("Energy\n")
                f.write(f"{config['energy']:.12f}\n")

            # Write stress
            if config.get('stress') and len(config['stress']) == 6:
                f.write("PlusStress: xx yy zz yz xz xy\n")
                stress_line = "  ".join(f"{val:>12.8f}" for val in config['stress'])
                f.write(f"  {stress_line}\n")

            f.write("END_CFG\n")

    print(f"Successfully wrote {len(configurations)} configurations to {output_file}")

In [ ]:
# =============================================================================
# OUTCAR TO CFG CONVERSION - USAGE EXAMPLE
# =============================================================================

# EXAMPLE: Convert VASP OUTCAR to MLIP CFG format
# UPDATE THESE PATHS to match your data location
outcar_path = '/Users/duygusenturk/Desktop/TU_Wien/DATA-VASP/6ps@2500K/Ti-v/NVT/25/OUTCAR'
output_training_file = '/Users/duygusenturk/Desktop/TU_Wien/configure files/6ps-2500K-Ti-v-25-NVT.cfg'

# Run the conversion
processed_configs = parse_outcar_to_mlip(outcar_path, output_training_file)
print(f"✓ Training file created with {processed_configs} configurations")

# EXAMPLE: Stress conversion verification
print("\n" + "="*50)
print("STRESS CONVERSION EXAMPLE")
print("="*50)

# Example values from VASP OUTCAR
xx_kb, yy_kb, zz_kb, xy_kb, yz_kb, zx_kb = -17.61542, 6.59015, -41.25430, -7.26889, -0.05273, -2.27008
cell_volume = 1605.94

# Convert kB to eV/Å³
conversion_factor = 160.2176  # kB to eV/Å³
xx_ev = xx_kb / conversion_factor
yy_ev = yy_kb / conversion_factor
zz_ev = zz_kb / conversion_factor
yz_ev = yz_kb / conversion_factor
zx_ev = zx_kb / conversion_factor
xy_ev = xy_kb / conversion_factor

# Multiply by cell volume to get eV
xx_eV_total = xx_ev * cell_volume
yy_eV_total = yy_ev * cell_volume
zz_eV_total = zz_ev * cell_volume
yz_eV_total = yz_ev * cell_volume
zx_eV_total = zx_ev * cell_volume
xy_eV_total = xy_ev * cell_volume

print(f'Cell Volume: {cell_volume:.2f} Å³')
print(f'Original stress (kB): XX={xx_kb:.3f}, YY={yy_kb:.3f}, ZZ={zz_kb:.3f}')
print(f'Converted stress (eV): XX={xx_eV_total:.6f}, YY={yy_eV_total:.6f}, ZZ={zz_eV_total:.6f}')
print("="*50)

---

# 2. Energy Convergence Analysis

## 2.1 Convergence Detection Functions

Analyze energy convergence during VASP calculations to determine the optimal point for data extraction.

### Features:
- **Automatic atom count detection** from OUTCAR
- **Customizable convergence criteria** (threshold and window size)
- **Visual convergence identification** with annotations
- **Per-atom energy analysis**

In [ ]:
# =============================================================================
# ENERGY CONVERGENCE ANALYSIS FUNCTIONS
# =============================================================================

def find_convergence_point(energies, threshold, window_size):
    """
    Find the convergence point where energy differences remain below threshold
    
    Parameters:
    energies (np.array): Array of energies per atom
    threshold (float): Convergence threshold (eV/atom)
    window_size (int): Number of consecutive steps to check
    
    Returns:
    int or None: Index of convergence point, None if not converged
    """
    if len(energies) < window_size + 1:
        return None
    
    for i in range(len(energies) - window_size):
        energy_diffs = np.abs(np.diff(energies[i:i+window_size+1]))
        if np.all(energy_diffs < threshold):
            return i
    
    return None

def plot_energy_convergence(outcar_file, energy_threshold=1e-6, window_size=10, natoms_manual=None):
    """
    Plot energy convergence from VASP OUTCAR file
    
    Parameters:
    outcar_file (str): Path to OUTCAR file
    energy_threshold (float): Convergence threshold (eV/atom)
    window_size (int): Number of consecutive steps for convergence
    natoms_manual (int): Manual override for number of atoms
    
    Returns:
    tuple: (time_steps, energies_per_atom, convergence_step)
    """
    try:
        with open(outcar_file, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: File {outcar_file} not found!")
        return None, None, None
    except Exception as e:
        print(f"Error reading file: {e}")
        return None, None, None
    
    # Extract energies and ionic steps
    energies = []
    ionic_steps = []
    natoms = natoms_manual

    # Auto-detect number of atoms if not provided
    if natoms is None:
        for line in lines:
            if "NIONS" in line:
                try:
                    if "number of ions     NIONS =" in line:
                        natoms = int(line.split("NIONS =")[1].strip().split()[0])
                        break
                    elif "NIONS =" in line and line.split("NIONS =")[1].strip():
                        parts = line.split("NIONS =")[1].strip().split()
                        if parts:
                            natoms = int(parts[0])
                            break
                except (ValueError, IndexError):
                    continue
        
        if natoms is None:
            for line in lines:
                if "ions per type" in line:
                    try:
                        counts = [int(x) for x in line.split()[4:]]
                        natoms = sum(counts)
                        print(f"Found natoms from 'ions per type': {natoms}")
                        break
                    except:
                        continue
    
    if natoms is None:
        print("Error: Could not determine number of atoms")
        print("Please provide natoms_manual parameter")
        return None, None, None
    
    print(f"Using {natoms} atoms {'(manual)' if natoms_manual else '(auto-detected)'}")
    
    # Extract energies from each ionic step
    current_step = 0
    for i, line in enumerate(lines):
        if "--------------------------------------- Ionic step" in line:
            try:
                current_step = int(line.split("Ionic step")[1].split("---")[0].strip())
            except:
                current_step += 1
        
        if "total energy   ETOTAL" in line and "eV" in line:
            try:
                parts = line.split("=")
                if len(parts) > 1:
                    energy = float(parts[1].split()[0])
                    energies.append(energy)
                    ionic_steps.append(current_step)
            except:
                pass
    
    if not energies:
        print("No energy data found in OUTCAR")
        return None, None, None
    
    print(f"Found {len(energies)} energy values")
    
    # Convert to energy per atom
    energies_per_atom = np.array(energies) / natoms
    time_steps = np.array(ionic_steps)
    
    # Find convergence point
    convergence_step = find_convergence_point(energies_per_atom, energy_threshold, window_size)
    
    # Create the plot
    plt.figure(figsize=(12, 8))
    plt.plot(time_steps, energies_per_atom, '-', linewidth=2.5, color='#FE420E', label='Energy per atom')
    plt.xlabel('Ionic Step', fontsize=12, fontweight='bold')
    plt.ylabel('Energy per Atom (eV/atom)', fontsize=12, fontweight='bold')
    plt.title('Energy Convergence Analysis', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.6, linestyle='-.', color='gray')
    
    # Mark convergence point if found
    if convergence_step is not None and convergence_step < len(time_steps):
        conv_x = time_steps[convergence_step]
        conv_y = energies_per_atom[convergence_step]
        
        plt.plot(conv_x, conv_y, 'ro', markersize=10, label=f'Convergence (Step {conv_x})')
        
        # Position annotation
        x_middle = (min(time_steps) + max(time_steps)) / 2
        y_bottom = min(energies_per_atom) + (max(energies_per_atom) - min(energies_per_atom)) * 0.1
        
        plt.annotate(f'Convergence\nStep {conv_x}\n{conv_y:.6f} eV/atom', 
                    xy=(conv_x, conv_y), 
                    xytext=(x_middle, y_bottom),
                    arrowprops=dict(arrowstyle='->', color='#C1F80A', lw=2),
                    fontsize=10, ha='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFFCB', alpha=0.7))
    
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"Total ionic steps: {len(time_steps)}")
    print(f"Initial energy per atom: {energies_per_atom[0]:.6f} eV/atom")
    print(f"Final energy per atom: {energies_per_atom[-1]:.6f} eV/atom")
    print(f"Energy change: {energies_per_atom[-1] - energies_per_atom[0]:.6f} eV/atom")
    
    if convergence_step is not None:
        print(f"Convergence achieved at step: {time_steps[convergence_step]}")
        print(f"Convergence energy: {energies_per_atom[convergence_step]:.6f} eV/atom")
    else:
        print(f"Convergence not achieved within threshold {energy_threshold} eV/atom")
    
    return time_steps, energies_per_atom, convergence_step

In [ ]:
# =============================================================================
# ENERGY CONVERGENCE ANALYSIS - USAGE EXAMPLE
# =============================================================================

# EXAMPLE: Analyze energy convergence
# UPDATE THIS PATH to your OUTCAR file
outcar_path = '/Users/duygusenturk/Downloads/OUTCAR'

# Run convergence analysis
result = plot_energy_convergence(
    outcar_path, 
    energy_threshold=1e-6,  # Convergence threshold (eV/atom)
    window_size=15,         # Number of consecutive steps to check
    natoms_manual=162       # Number of atoms (None for auto-detection)
)

if result[0] is not None:
    time_steps, energies_per_atom, convergence_step = result
    print("✓ Energy convergence analysis completed successfully")
    
    if convergence_step is not None:
        print(f"✓ System converged at step {time_steps[convergence_step]}")
        print(f"  Recommendation: Use configurations from step {time_steps[convergence_step]} onwards")
    else:
        print("⚠ System has not converged within the specified criteria")
        print("  Consider using a larger threshold or analyzing the full trajectory")
else:
    print("✗ Convergence analysis failed - check file path and format")

---

# 3. Data Filtering and Management

## 3.1 CFG File Filtering Functions

Filter CFG files to remove unconverged configurations and manage large datasets.

In [ ]:
# =============================================================================
# DATA FILTERING FUNCTIONS
# =============================================================================

def filter_cfg_file(input_cfg_file, output_cfg_file, skip_first_n=0):
    """
    Filter CFG file by skipping the first N configurations
    
    Parameters:
    input_cfg_file (str): Path to input CFG file
    output_cfg_file (str): Path to output filtered CFG file
    skip_first_n (int): Number of initial configurations to skip
    
    Returns:
    tuple: (total_configs, kept_configs)
    """
    with open(input_cfg_file, 'r') as f:
        lines = f.readlines()
    
    # Find all BEGIN_CFG positions
    cfg_starts = []
    for i, line in enumerate(lines):
        if line.strip() == "BEGIN_CFG":
            cfg_starts.append(i)
    
    total_configs = len(cfg_starts)
    print(f"Found {total_configs} configurations, skipping first {skip_first_n}")
    
    if skip_first_n >= total_configs:
        print(f"Error: Cannot skip {skip_first_n} configs from {total_configs} total")
        return total_configs, 0
    
    # Find where to start writing
    start_writing_from = cfg_starts[skip_first_n] if skip_first_n < len(cfg_starts) else len(lines)
    
    # Write remaining configurations
    with open(output_cfg_file, 'w') as f:
        f.writelines(lines[start_writing_from:])
    
    kept_configs = total_configs - skip_first_n
    print(f"Kept {kept_configs} configurations in {output_cfg_file}")
    
    return total_configs, kept_configs

def merge_cfg_files(input_folder, output_file, pattern="*.cfg"):
    """
    Merge all CFG files in a folder into one file
    
    Parameters:
    input_folder (str): Path to folder containing CFG files
    output_file (str): Path to output merged file
    pattern (str): File pattern to match (default: "*.cfg")
    
    Returns:
    int: Total number of configurations merged
    """
    # Get all matching files
    file_pattern = os.path.join(input_folder, pattern)
    cfg_files = sorted(glob.glob(file_pattern))
    
    if not cfg_files:
        print(f"No files found matching {file_pattern}")
        return 0
    
    print(f"Found {len(cfg_files)} files to merge:")
    for f in cfg_files:
        print(f"  - {os.path.basename(f)}")
    
    total_configs = 0
    
    with open(output_file, 'w') as outf:
        for cfg_file in cfg_files:
            print(f"Processing {os.path.basename(cfg_file)}...")
            
            with open(cfg_file, 'r') as inf:
                content = inf.read()
                outf.write(content)
                
                # Count configurations
                configs_in_file = content.count('BEGIN_CFG')
                total_configs += configs_in_file
                print(f"  Added {configs_in_file} configurations")
    
    print(f"\nMerged {len(cfg_files)} files into {output_file}")
    print(f"Total configurations: {total_configs}")
    
    return total_configs

In [ ]:
# =============================================================================
# DATA FILTERING AND MANAGEMENT - USAGE EXAMPLES
# =============================================================================

# EXAMPLE 1: Filter CFG file to remove unconverged configurations
input_cfg = '/Users/duygusenturk/Desktop/TU_Wien/configure files/6ps-2500K-B-i-70-NVT.cfg'
output_cfg = '/Users/duygusenturk/Desktop/TU_Wien/configure files/6ps-2500K-B-i-70-NVT-filtered.cfg'

# Skip first 1000 configurations (example: first 1000 steps are unconverged)
total, kept = filter_cfg_file(input_cfg, output_cfg, skip_first_n=1000)
print(f"✓ Filtered: {total} → {kept} configurations")

print("\n" + "="*50)

# EXAMPLE 2: Merge multiple CFG files
input_folder = "/Users/duygusenturk/Desktop/TU_Wien/configure files"
output_file = "/Users/duygusenturk/Desktop/TU_Wien/configure files/all_merged.cfg"

# Uncomment to merge all .cfg files
# total_merged = merge_cfg_files(input_folder, output_file)
# print(f"✓ Merged dataset contains {total_merged} total configurations")

print("Update the file paths and uncomment the merge operation to combine datasets")

print("\n" + "="*50)
print("TERMINAL COMMANDS FOR MERGING")
print("="*50)
print("# Navigate to your configure files folder:")
print("cd '/Users/duygusenturk/Desktop/TU_Wien/configure files'")
print()
print("# Merge all .cfg files:")
print("cat *.cfg > all_merged.cfg")
print()
print("# Merge files in specific order:")
print("cat file1.cfg file2.cfg file3.cfg > merged_file.cfg")
print()
print("# Count total configurations in merged file:")
print("grep -c 'BEGIN_CFG' all_merged.cfg")

---

# 4. MLIP Grade Analysis

## 4.1 Grade Distribution Analysis

Analyze MLIP sampling grades to understand model uncertainty and data quality.

### Features:
- **Grade extraction** from MLIP output files
- **Statistical analysis** (mean, std, percentiles)
- **Visual histograms** with customizable bins
- **Quality assessment** (grades > 2.0 indicate high uncertainty)

In [ ]:
# =============================================================================
# MLIP GRADE ANALYSIS FUNCTIONS
# =============================================================================

def parse_grades_from_file(file_path):
    """
    Parse sampling grades from MLIP output file
    
    Parameters:
    file_path (str): Path to MLIP output file containing grades
    
    Returns:
    list: List of grade values
    """
    grades = []
    found_start = False
    
    try:
        with open(file_path, 'r') as f:
            for line in f:
                if "Starting grades evaluation" in line:
                    found_start = True
                    print("Found 'Starting grades evaluation' - collecting grades...")
                    continue
                
                if found_start and "grade:" in line:
                    match = re.search(r'grade:\s*([0-9.]+)', line)
                    if match:
                        grade = float(match.group(1))
                        grades.append(grade)
    
    except FileNotFoundError:
        print(f"Error: File {file_path} not found!")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []
    
    print(f"Collected {len(grades)} grade values")
    return grades

def create_grade_histogram(grades, bins=20, title="Grade Distribution", save_path=None):
    """
    Create histogram of MLIP grades with statistical analysis
    
    Parameters:
    grades (list): List of grade values
    bins (int): Number of histogram bins
    title (str): Plot title
    save_path (str): Path to save histogram (optional)
    
    Returns:
    tuple: (grades, bin_edges, frequencies)
    """
    if not grades:
        print("No grades to plot!")
        return None, None, None
    
    # Calculate statistics
    mean_grade = np.mean(grades)
    std_grade = np.std(grades)
    min_grade = np.min(grades)
    max_grade = np.max(grades)
    
    # Count grades higher than 2 (high uncertainty threshold)
    grades_above_2 = np.sum(np.array(grades) > 2)
    percentage_above_2 = (grades_above_2 / len(grades)) * 100
    
    # Create histogram
    plt.figure(figsize=(12, 8))
    
    # Plot histogram
    n, bins_edges, patches = plt.hist(grades, bins=bins, alpha=0.7, color='#FE420E', 
                                     edgecolor='black', linewidth=1.2)
    
    # Styling
    plt.xlabel('Grade Value', fontsize=14, fontweight='bold')
    plt.ylabel('Frequency', fontsize=14, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, linestyle='-.', color='gray')
    
    # Add statistics text box
    stats_text = f'Total samples: {len(grades)}\n'
    stats_text += f'Mean: {mean_grade:.3f}\n'
    stats_text += f'Std Dev: {std_grade:.3f}\n'
    stats_text += f'Min: {min_grade:.3f}\n'
    stats_text += f'Max: {max_grade:.3f}\n'
    stats_text += f'Grades > 2.0: {grades_above_2} ({percentage_above_2:.1f}%)'
    
    plt.text(0.98, 0.98, stats_text, transform=plt.gca().transAxes, 
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
             fontsize=10)
    
    # Highlight the bin with highest frequency
    max_freq_idx = np.argmax(n)
    patches[max_freq_idx].set_color('#C1F80A')
    patches[max_freq_idx].set_alpha(0.9)
    
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Histogram saved to: {save_path}")
    
    plt.show()
    
    # Print detailed analysis
    print("\n" + "="*50)
    print("INTERVAL ANALYSIS")
    print("="*50)
    
    for i in range(len(bins_edges)-1):
        bin_start = bins_edges[i]
        bin_end = bins_edges[i+1]
        count = int(n[i])
        percentage = (count / len(grades)) * 100
        
        marker = " ← PEAK" if i == max_freq_idx else ""
        print(f"[{bin_start:.3f} - {bin_end:.3f}]: {count:4d} samples ({percentage:5.1f}%){marker}")
    
    # Print summary statistics
    print("\n" + "="*50)
    print("GRADE SUMMARY")
    print("="*50)
    print(f"Total samples: {len(grades)}")
    print(f"Grades ≤ 2.0: {len(grades) - grades_above_2} ({100 - percentage_above_2:.1f}%)")
    print(f"Grades > 2.0: {grades_above_2} ({percentage_above_2:.1f}%)")
    print("\nInterpretation:")
    print("- Grades ≤ 2.0: Well-represented by current model")
    print("- Grades > 2.0: High uncertainty, consider for active learning")
    print("="*50)
    
    return grades, bins_edges, n

def analyze_grade_file(file_path, bins=20, save_histogram=True):
    """
    Complete grade analysis workflow
    
    Parameters:
    file_path (str): Path to MLIP output file
    bins (int): Number of histogram bins
    save_histogram (bool): Whether to save histogram image
    
    Returns:
    tuple: Analysis results (grades, bin_edges, frequencies)
    """
    print(f"Analyzing grades from: {file_path}")
    
    # Extract grades
    grades = parse_grades_from_file(file_path)
    
    if not grades:
        return None
    
    # Create output filename for histogram
    save_path = None
    if save_histogram:
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        save_path = f"/Users/duygusenturk/Desktop/TU_Wien/{base_name}_grade_histogram.png"
    
    # Create histogram
    title = f"Grade Distribution ({len(grades)} samples)"
    results = create_grade_histogram(grades, bins=bins, title=title, save_path=save_path)
    
    return results

In [ ]:
# =============================================================================
# MLIP GRADE ANALYSIS - USAGE EXAMPLE
# =============================================================================

# EXAMPLE: Analyze MLIP grades and create histogram
# UPDATE THIS PATH to your grades file
file_path = "/Volumes/DATA/grades.txt"

# Run complete grade analysis
results = analyze_grade_file(file_path, bins=25, save_histogram=True)

if results:
    grades, bin_edges, frequencies = results
    print(f"✓ Processed {len(grades)} grades successfully!")
    
    # Additional analysis
    high_uncertainty = [g for g in grades if g > 2.0]
    if high_uncertainty:
        print(f"\n📊 Quality Assessment:")
        print(f"   - {len(high_uncertainty)} configurations have high uncertainty (grade > 2.0)")
        print(f"   - These represent {len(high_uncertainty)/len(grades)*100:.1f}% of total data")
        print(f"   - Consider adding similar structures to training set")
    else:
        print(f"\n✓ All grades ≤ 2.0: Model shows good confidence across all sampled configurations")
else:
    print("✗ Grade analysis failed - check file path and format")

---

# 5. Tips and Best Practices

## Common Workflows

### 1. **New VASP → MLIP Conversion**
```python
# Step 1: Analyze convergence
plot_energy_convergence('OUTCAR', energy_threshold=1e-6)

# Step 2: Convert OUTCAR → CFG (skip unconverged steps)
parse_outcar_to_mlip('OUTCAR', 'output.cfg')

# Step 3: Filter if needed
filter_cfg_file('output.cfg', 'filtered.cfg', skip_first_n=100)
```

### 2. **Merge Multiple Datasets**
```python
# Option A: Python function
merge_cfg_files('/path/to/cfg/files/', 'merged.cfg')

# Option B: Terminal command
# cd /path/to/cfg/files/
# cat *.cfg > merged.cfg
```

### 3. **Grade Analysis Workflow**
```python
# Analyze model uncertainty
analyze_grade_file('grades.txt', bins=30)

# High grades (>2.0) indicate need for more training data
```

## Troubleshooting

### **File Reading Issues**
- Check file paths are correct and files exist
- Ensure OUTCAR files are complete (not truncated)
- Verify file permissions for reading/writing

### **Stress Conversion Problems**
- Check if cell volume is properly extracted
- Verify stress tensor format in OUTCAR
- Conversion factor: 1 eV/Å³ = 160.2176 kB

### **Memory Issues**
- Process large OUTCAR files in chunks
- Filter configurations early in the workflow
- Use convergence analysis to reduce dataset size

### **Atom Type Detection**
- Manually specify `natoms_manual` if auto-detection fails
- Check POTCAR section in OUTCAR for element information
- Verify "ions per type" line exists

## File Format Notes

### **CFG Format Structure**
```
BEGIN_CFG
Size
<number_of_atoms>
Supercell
<lattice_vector_1>
<lattice_vector_2>
<lattice_vector_3>
AtomData: id type cartes_x cartes_y cartes_z fx fy fz
<atom_data_lines>
Energy
<total_energy_in_eV>
PlusStress: xx yy zz yz xz xy
<stress_tensor_in_eV>
END_CFG
```

### **Stress Tensor Conversion**
- **VASP**: kB units, order XX YY ZZ XY YZ ZX
- **MLIP**: eV units, order XX YY ZZ YZ XZ XY
- **Conversion**: (kB_value / 160.2176) × cell_volume = eV_value

---

**Happy MLIP training! 🚀**